# Colab Training Notebook

## How to run

1. Change runtime to T4 GPU (Runtime → Change runtime type).
2. Run Cell 0 first. The kernel will be killed and Colab will reconnect.
3. After reconnection, SKIP Cell 0 and run Cells 1–8 in order.

In [ ]:
# Cell 0 — Force-install pinned dependencies and restart kernel.
# Run this FIRST. The kernel will die after the os.kill line; Colab will
# automatically reconnect. When it does, skip Cell 0 and run from Cell 1.

import subprocess, sys, os

# Force-reinstall datasets at the pinned major version. --force-reinstall
# overrides Colab's preinstalled 4.x; --no-deps avoids resolver picking 4.x back up.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--force-reinstall", "--no-deps", "datasets<4.0"],
    check=True,
)

# Safety net: upgrade torchvision so that if datasets' torch formatter ever does
# `from torchvision.io import VideoReader`, the symbol exists. The real fix is in
# src/train.py (no set_format("torch")), so this import should never fire.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "torchvision"],
    check=True,
)

# Workaround: Colab ships torchaudio (CUDA 12.8) incompatible with PyTorch (CUDA 13.0).
# transformers imports torchaudio for RNN-T loss which we don't use. Removing it
# makes the Trainer import path skip the broken module.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchaudio"],
    check=True,
)

# Verify the install took before we kill the kernel.
import datasets
assert datasets.__version__.startswith("3."), (
    f"Expected datasets 3.x, got {datasets.__version__}. "
    "Pin did not take effect; investigate before restarting."
)
print(f"datasets {datasets.__version__} installed. Killing kernel for clean reimport...")

# Hard kernel restart. Colab will auto-reconnect within a few seconds.
os.kill(os.getpid(), 9)

In [ ]:
# Cell 1: confirm a GPU is attached.
!nvidia-smi
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime > Change runtime type > "
    "Hardware accelerator > GPU (T4), then re-run this cell."
)
print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 2: clone the repo and cd into it.
!git clone https://github.com/ashirbaydaulet/paper-surgeon.git
%cd paper-surgeon

In [ ]:
# Cell 3: install dependencies. accelerate>=1.1.0 is required by the HF Trainer
# but is not pinned in requirements.txt, so install it explicitly.
!pip install -q -r requirements.txt
!pip install -q "accelerate>=1.1.0"

In [ ]:
# Cell 4: mount Drive and create the model-artifact directory.
import os

from google.colab import drive

drive.mount('/content/drive')
MODELS_DIR = "/content/drive/MyDrive/paper-surgeon-models"
os.makedirs(MODELS_DIR, exist_ok=True)
print("Saving model artifacts to:", MODELS_DIR)

In [ ]:
# Cell 5: log in to Weights & Biases. Paste your API key when prompted.
import wandb

wandb.login()

In [ ]:
# Cell 6: GPU sanity check — 1 epoch of SciBERT on 1000 train examples.
# Should finish in ~2-3 minutes on a T4. Confirms the full pipeline works on GPU
# before launching the real sweep.
!python -m src.train --model scibert --epochs 1 --subset 1000 --batch-size 32 \
  --output-dir /content/drive/MyDrive/paper-surgeon-models/smoke_gpu \
  --wandb-project paper-surgeon --wandb-run-name smoke_gpu

In [ ]:
# Cell 7: sweep with resumability. Calls src.train.main() in-process so Drive
# auth and W&B login persist across runs and one crash doesn't abort the sweep.
import os
import sys
import traceback

import torch

from src import train

MODELS_DIR = "/content/drive/MyDrive/paper-surgeon-models"

# ── Sweep plan ───────────────────────────────────────────────────────────────
SEEDS = [42, 123]
SCIBERT_LRS = [1e-5, 2e-5, 5e-5]

# Phase 1: find the best SciBERT LR using a single seed (42) -> 3 runs.
configs = [("scibert", lr, 42) for lr in SCIBERT_LRS]

# Phase 2 (run later, after Phase 1 reveals the best LR):
# TODO: set BEST_LR to the winning SciBERT LR from Phase 1, then uncomment.
#   BEST_LR = 2e-5  # <-- fill in from Phase 1 results
#   configs += [("scibert", BEST_LR, 123)]                       # SciBERT, 2nd seed
#   configs += [("bert-base", BEST_LR, seed) for seed in SEEDS]  # baseline, both seeds

def out_dir_for(model, lr, seed):
    return f"{MODELS_DIR}/{model}_lr{lr}_seed{seed}"

def is_done(model, lr, seed):
    return os.path.exists(os.path.join(out_dir_for(model, lr, seed), "run_metadata.json"))

# ── Progress note ────────────────────────────────────────────────────────────
n_total = len(configs)
n_done = sum(is_done(*c) for c in configs)
print(
    f"Planned this session: {n_total} | already done in Drive: {n_done} | "
    f"remaining: {n_total - n_done}"
)

# ── Run loop ─────────────────────────────────────────────────────────────────
attempted = 0
failures = []
for i, (model, lr, seed) in enumerate(configs, 1):
    run_name = f"{model}-lr{lr}-seed{seed}"
    out_dir = out_dir_for(model, lr, seed)
    if is_done(model, lr, seed):
        print(f"[{i}/{n_total}] SKIP (already done): {run_name}")
        continue
    attempted += 1
    sys.argv = [
        "src.train",
        "--model", model,
        "--learning-rate", str(lr),
        "--batch-size", "32",
        "--epochs", "2",
        "--max-length", "128",
        "--weight-decay", "0.01",
        "--warmup-ratio", "0.1",
        "--seed", str(seed),
        "--output-dir", out_dir,
        "--wandb-project", "paper-surgeon",
        "--wandb-run-name", run_name,
    ]
    print(f"\n{'='*70}\n[{i}/{n_total}] {run_name}\n{'='*70}")
    try:
        train.main()
    except Exception as exc:
        traceback.print_exc()
        failures.append((run_name, repr(exc)))
        print(f"!! FAILED: {run_name} -> {exc}")
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\n=== Sweep complete ===")
print(f"Succeeded this session: {attempted - len(failures)}/{attempted}")
for name, err in failures:
    print(f"  FAILED {name}: {err}")

In [ ]:
# Cell 8: aggregate every run_metadata.json into one table and save to Drive.
# Smoke-test runs (config.subset is not None) are skipped so only full runs land
# in the results table.
import json
import os

import pandas as pd

MODELS_DIR = "/content/drive/MyDrive/paper-surgeon-models"
PER_CLASS = [
    "test_f1_BACKGROUND", "test_f1_OBJECTIVE", "test_f1_METHODS",
    "test_f1_RESULTS", "test_f1_CONCLUSIONS",
]

rows = []
for root, _, files in os.walk(MODELS_DIR):
    if "run_metadata.json" not in files:
        continue
    with open(os.path.join(root, "run_metadata.json")) as f:
        meta = json.load(f)
    cfg, m = meta["config"], meta["test_metrics"]
    if cfg.get("subset") is not None:
        continue  # skip smoke-test runs
    rows.append({
        "model": cfg["model"],
        "learning_rate": cfg["learning_rate"],
        "seed": cfg["seed"],
        "test_macro_f1": m.get("test_macro_f1"),
        "test_accuracy": m.get("test_accuracy"),
        **{k: m.get(k) for k in PER_CLASS},
        "train_time_seconds": meta.get("train_time_seconds"),
    })

df = (
    pd.DataFrame(rows)
    .sort_values(["model", "learning_rate", "seed"])
    .reset_index(drop=True)
)
csv_path = f"{MODELS_DIR}/sweep_results.csv"
df.to_csv(csv_path, index=False)
print("Wrote:", csv_path)
print(df.to_string(index=False))